# 🌐 Google Colab Zero-Cost OCR API Server for Antigravity & Local Clients

This notebook turns your Google Colab free GPU into a **live, remote REST API endpoint**.
You can send PDFs from your local PC (or Antigravity) to this Colab server and receive converted Markdown back instantly!

In [ ]:
# Step 1: Install Server Dependencies
!pip install -q fastapi uvicorn pyngrok nest-asyncio pymupdf transformers torch accelerate pillow

In [ ]:
# Step 2: Load Model (GLM-OCR ~0.9B)
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

model_id = "zai-org/GLM-OCR"
processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
print("Model loaded successfully in GPU VRAM!")

In [ ]:
# Step 3: Launch FastAPI & Expose Public URL via LocalTunnel
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import uvicorn
import threading
import io
import pymupdf as fitz
from PIL import Image

app = FastAPI(title="Colab OCR Engine")

@app.post("/parse")
async def parse_endpoint(file: UploadFile = File(...)):
    contents = await file.read()
    doc = fitz.open(stream=contents, filetype="pdf")
    page_mds = []
    
    for idx, page in enumerate(doc):
        pix = page.get_pixmap(dpi=150)
        img = Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Extract all text, sections, and tables verbatim into Markdown."}
                ]
            }
        ]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=4096)
        prompt_len = inputs["input_ids"].shape[-1]
        text = processor.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
        page_mds.append(text)
        
    return {"status": "success", "pages": len(doc), "markdown": "\n\n---\n\n".join(page_mds)}

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

threading.Thread(target=run_server, daemon=True).start()
print("Server running on port 8000.")

# Expose via localtunnel
!npx localtunnel --port 8000